# 01 — Is the CLIP text encoder MDM conditions on blind to spatial language?

**Question.** MDM's text conditioning is a frozen CLIP ViT-B/32 text encoder (`model/mdm.py`,
`load_and_freeze_clip`). Does that encoder actually distinguish spatial/directional language
("left" vs "right", "forward" vs "backward") at all, or does it embed such minimal pairs as
near-identical — which would mean the *conditioning signal itself* cannot tell the diffusion
model which direction was asked for, independent of anything about the generator or the dataset?

**Why this matters here.** Per `docs/DECISIONS.md` D-29: before running any more training
comparisons (which repeat E1's exact shape — an unproven training budget, scored by an instrument
already under suspicion), the instruments themselves need checking. This notebook checks one
candidate root cause for the diffusion model's own weak-conditioning behavior observed all
session (`docs/EXPERIMENT_LOG.md` E1B) — not by training anything, by directly probing the frozen
encoder used in the real pipeline.

**Design — three groups, not two (revised after review SUP-20260907-82).** The original design
compared spatial-modifier minimal pairs (left/right, forward/backward) against non-spatial
*verb* minimal pairs (walks/runs, sits/stands). That confounds two different hypotheses: "CLIP is
blind to spatial language specifically" and the weaker, less interesting "CLIP separates verbs
better than modifiers in general" — since a spatial term and a verb differ in more ways than just
spatial-ness. **A third arm isolates this:** non-spatial *modifier* pairs, same syntactic slot as
the spatial pairs (an adjective or adverb substitution — color, size, manner, condition — never
directional). If spatial modifiers are under-separated relative to non-spatial *modifiers*
specifically, the spatial claim is isolated from the verb-vs-modifier confound. If the two
modifier groups look the same, the honest finding is "CLIP under-separates modifiers generally,
spatial included" — a real limitation either way, reported as whichever one the data supports.

**Instrument.** MDM's own CLIP loader and tokenization convention, copied verbatim from
`model/mdm.py::MDM.load_and_freeze_clip` / `MDM.clip_encode_text` — not reimplemented, not a
stand-in. `ViT-B/32`, the version `utils/model_util.py` hardcodes for every checkpoint in this
project.

**Literature context** (`guidance/VERIFICATION_NOTE.md`, independently re-verified before this
notebook was written): arXiv:2311.11477, "What's left can't be right — the remaining positional
incompetence of contrastive vision-language models," confirms the *direction* of this finding for
CLIP-style models generally. The "BLIP scored 56% vs 99% human" statistic sometimes attached to
that paper belongs to a different paper (**not cited here** — it does not survive
`VERIFICATION_NOTE.md`'s own re-check).

In [1]:
import os
import sys

MDM_ROOT = os.path.join("..", "third_party", "motion-diffusion-model")
sys.path.insert(0, os.path.abspath(MDM_ROOT))

import numpy as np
import torch
import clip
from scipy import stats

torch.manual_seed(0)
print("torch:", torch.__version__)
print("clip available models:", clip.available_models())

torch: 2.13.0
clip available models: ['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']


## 1. Load MDM's own CLIP text encoder, exactly as `model/mdm.py` does

`clip_version = 'ViT-B/32'` is hardcoded in `utils/model_util.py::get_model_args` for every
checkpoint this project uses — not a choice made here. `load_and_freeze_clip` and
`clip_encode_text` below are copied verbatim from `model/mdm.py` (device `'cpu'`, `jit=False`,
frozen weights, `max_text_len=20` -> `context_length=22` -> zero-padded to 77), so the tokenization
and truncation behavior matches the real pipeline exactly, not an approximation of it.

In [2]:
def load_and_freeze_clip(clip_version):
    # Copied verbatim from third_party/motion-diffusion-model/model/mdm.py::MDM.load_and_freeze_clip
    clip_model, clip_preprocess = clip.load(clip_version, device='cpu', jit=False)
    clip.model.convert_weights(clip_model)
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad = False
    return clip_model


def clip_encode_text(clip_model, raw_text, dataset="humanml"):
    # Copied verbatim from third_party/motion-diffusion-model/model/mdm.py::MDM.clip_encode_text
    # (device fixed to cpu here; the real pipeline uses next(self.parameters()).device, but the
    # text encoder's own weights and tokenization are identical regardless of device).
    device = "cpu"
    max_text_len = 20 if dataset in ["humanml", "kit"] else None
    if max_text_len is not None:
        default_context_length = 77
        context_length = max_text_len + 2
        assert context_length < default_context_length
        texts = clip.tokenize(raw_text, context_length=context_length, truncate=True).to(device)
        zero_pad = torch.zeros([texts.shape[0], default_context_length - context_length],
                                dtype=texts.dtype, device=texts.device)
        texts = torch.cat([texts, zero_pad], dim=1)
    else:
        texts = clip.tokenize(raw_text, truncate=True).to(device)
    with torch.no_grad():
        return clip_model.encode_text(texts).float()


clip_model = load_and_freeze_clip("ViT-B/32")
print("CLIP text encoder loaded and frozen.")

CLIP text encoder loaded and frozen.


## 2. Minimal pairs — three matched groups

- **Spatial:** substituted word is a directional/positional modifier (left/right,
  forward/backward, clockwise/counterclockwise, in front of/behind).
- **Non-spatial verb (original control):** substituted word is a verb.
- **Non-spatial modifier (new third arm, SUP-82):** substituted word is a modifier
  (adjective/adverb) in the same syntactic slot as the spatial group — never directional.

In [3]:
SPATIAL_PAIRS = [
    ("a person raises their left arm", "a person raises their right arm"),
    ("a person turns left", "a person turns right"),
    ("a person walks forward", "a person walks backward"),
    ("a person steps over the box", "a person steps around the box"),
    ("a person kicks with their left leg", "a person kicks with their right leg"),
    ("a person moves forward", "a person moves backward"),
    ("a person walks to the left", "a person walks to the right"),
    ("a person raises their left hand", "a person raises their right hand"),
    ("a person leans forward", "a person leans backward"),
    ("a person jumps forward", "a person jumps backward"),
    ("a person waves with their left hand", "a person waves with their right hand"),
    ("a person spins clockwise", "a person spins counterclockwise"),
    ("a person steps forward", "a person steps backward"),
    ("a person reaches with their left arm", "a person reaches with their right arm"),
    ("a person walks in front of the chair", "a person walks behind the chair"),
    ("a person bends their left knee", "a person bends their right knee"),
]

NONSPATIAL_VERB_PAIRS = [
    ("a person walks forward", "a person runs forward"),
    ("a person sits on a chair", "a person stands on a chair"),
    ("a person waves with their hand", "a person claps with their hand"),
    ("a person kicks the ball", "a person throws the ball"),
    ("a person picks up the box", "a person drops the box"),
    ("a person walks quickly", "a person walks slowly"),
    ("a person jumps once", "a person jumps twice"),
    ("a person sits on the chair", "a person sits on the table"),
    ("a person claps their hands", "a person waves their hands"),
    ("a person walks forward", "a person skips forward"),
    ("a person reads a book", "a person writes a letter"),
    ("a person opens the door", "a person closes the door"),
    ("a person drinks water", "a person eats food"),
    ("a person plays the guitar", "a person plays the piano"),
    ("a person laughs loudly", "a person cries softly"),
    ("a person dances happily", "a person dances sadly"),
]

# Third arm (SUP-20260907-82): non-spatial MODIFIER pairs -- same syntactic slot as the spatial
# group (a single adjective/adverb substitution), so this isolates "spatial" from "modifier vs
# verb" as the source of any under-separation found above.
NONSPATIAL_MODIFIER_PAIRS = [
    ("a person raises their arm slowly", "a person raises their arm quickly"),
    ("a person walks slowly", "a person walks quickly"),
    ("a person raises their broken arm", "a person raises their injured arm"),
    ("a person kicks the red ball", "a person kicks the blue ball"),
    ("a person wears a blue shirt", "a person wears a red shirt"),
    ("a person picks up the small box", "a person picks up the large box"),
    ("a person carries a heavy bag", "a person carries a light bag"),
    ("a person claps their hands loudly", "a person claps their hands softly"),
    ("a person sits on the wooden chair", "a person sits on the metal chair"),
    ("a person throws the heavy ball", "a person throws the light ball"),
    ("a person dances gracefully", "a person dances clumsily"),
    ("a person raises their tired arm", "a person raises their strong arm"),
    ("a person walks on the wet floor", "a person walks on the dry floor"),
    ("a person holds the empty cup", "a person holds the full cup"),
    ("a person wears an old hat", "a person wears a new hat"),
    ("a person speaks quietly", "a person speaks loudly"),
]

print(f"{len(SPATIAL_PAIRS)} spatial pairs, {len(NONSPATIAL_VERB_PAIRS)} non-spatial-verb pairs, "
      f"{len(NONSPATIAL_MODIFIER_PAIRS)} non-spatial-modifier pairs")

16 spatial pairs, 16 non-spatial-verb pairs, 16 non-spatial-modifier pairs


In [4]:
def cosine_sim(a, b):
    a = a / a.norm(dim=-1, keepdim=True)
    b = b / b.norm(dim=-1, keepdim=True)
    return float((a * b).sum(dim=-1))


def pair_similarities(pairs):
    sims = []
    for s1, s2 in pairs:
        e1 = clip_encode_text(clip_model, [s1])
        e2 = clip_encode_text(clip_model, [s2])
        sims.append(cosine_sim(e1[0], e2[0]))
    return np.array(sims)


spatial_sims = pair_similarities(SPATIAL_PAIRS)
verb_sims = pair_similarities(NONSPATIAL_VERB_PAIRS)
modifier_sims = pair_similarities(NONSPATIAL_MODIFIER_PAIRS)

for (s1, s2), sim in zip(SPATIAL_PAIRS, spatial_sims):
    print(f"{sim:.4f}  SPATIAL           {s1!r} <-> {s2!r}")
print()
for (s1, s2), sim in zip(NONSPATIAL_VERB_PAIRS, verb_sims):
    print(f"{sim:.4f}  NONSPATIAL-VERB   {s1!r} <-> {s2!r}")
print()
for (s1, s2), sim in zip(NONSPATIAL_MODIFIER_PAIRS, modifier_sims):
    print(f"{sim:.4f}  NONSPATIAL-MOD    {s1!r} <-> {s2!r}")

0.9699  SPATIAL           'a person raises their left arm' <-> 'a person raises their right arm'
0.9630  SPATIAL           'a person turns left' <-> 'a person turns right'
0.9429  SPATIAL           'a person walks forward' <-> 'a person walks backward'
0.9750  SPATIAL           'a person steps over the box' <-> 'a person steps around the box'
0.9910  SPATIAL           'a person kicks with their left leg' <-> 'a person kicks with their right leg'
0.9489  SPATIAL           'a person moves forward' <-> 'a person moves backward'
0.9791  SPATIAL           'a person walks to the left' <-> 'a person walks to the right'
0.9723  SPATIAL           'a person raises their left hand' <-> 'a person raises their right hand'
0.9330  SPATIAL           'a person leans forward' <-> 'a person leans backward'
0.9478  SPATIAL           'a person jumps forward' <-> 'a person jumps backward'
0.9835  SPATIAL           'a person waves with their left hand' <-> 'a person waves with their right hand'
0.9637  SPAT

## 3. Three distributions, and the disentangling test

If the spatial blind spot is real and specific to spatial-ness (not just "modifiers separate
worse than verbs"), `spatial_sims` should sit measurably **higher** (less separated) than
`modifier_sims` specifically — the same syntactic slot, only the spatial-ness differs. The
verb comparison is kept for continuity with the original design, but **the spatial-vs-modifier
comparison is now the load-bearing one.**

In [5]:
def describe(name, arr):
    print("%-20s (n=%2d): mean=%.4f  std=%.4f  min=%.4f  max=%.4f" % (
        name, len(arr), arr.mean(), arr.std(ddof=1), arr.min(), arr.max()))

describe("Spatial", spatial_sims)
describe("Nonspatial-verb", verb_sims)
describe("Nonspatial-modifier", modifier_sims)

# Overall omnibus test across all three groups (no assumption of normality)
h_stat, h_p = stats.kruskal(spatial_sims, verb_sims, modifier_sims)
print(f"\nKruskal-Wallis (all 3 groups): H={h_stat:.3f}, p={h_p:.5f}")

def compare(name_a, a, name_b, b, alt="two-sided"):
    t_stat, t_p = stats.ttest_ind(a, b, equal_var=False, alternative=alt)
    u_stat, u_p = stats.mannwhitneyu(a, b, alternative=alt)
    n1, n2 = len(a), len(b)
    rank_biserial = (2 * u_stat) / (n1 * n2) - 1  # positive => group a stochastically larger
    print(f"\n{name_a} vs {name_b}:")
    print(f"  Welch's t-test:  t={t_stat:.3f}, p={t_p:.5f} (alternative={alt})")
    print(f"  Mann-Whitney U:  U={u_stat:.1f}, p={u_p:.5f} (alternative={alt})")
    print(f"  Rank-biserial effect size: {rank_biserial:+.3f}")
    return t_p, u_p, rank_biserial

print("=== Original comparison (spatial vs verb) ===")
_, u_p_verb, rb_verb = compare("Spatial", spatial_sims, "Nonspatial-verb", verb_sims, alt="greater")

print("\n=== Load-bearing comparison (spatial vs same-slot modifier) ===")
_, u_p_modifier, rb_modifier = compare("Spatial", spatial_sims, "Nonspatial-modifier", modifier_sims, alt="greater")

print("\n=== Check: do the two non-spatial groups differ from each other? ===")
_, u_p_verb_vs_mod, rb_verb_vs_mod = compare("Nonspatial-verb", verb_sims, "Nonspatial-modifier", modifier_sims, alt="two-sided")

Spatial              (n=16): mean=0.9654  std=0.0167  min=0.9330  max=0.9910
Nonspatial-verb      (n=16): mean=0.9296  std=0.0312  min=0.8759  max=0.9762
Nonspatial-modifier  (n=16): mean=0.9446  std=0.0400  min=0.8131  max=0.9890

Kruskal-Wallis (all 3 groups): H=11.419, p=0.00331
=== Original comparison (spatial vs verb) ===

Spatial vs Nonspatial-verb:
  Welch's t-test:  t=4.046, p=0.00025 (alternative=greater)
  Mann-Whitney U:  U=215.0, p=0.00056 (alternative=greater)
  Rank-biserial effect size: +0.680

=== Load-bearing comparison (spatial vs same-slot modifier) ===

Spatial vs Nonspatial-modifier:
  Welch's t-test:  t=1.917, p=0.03481 (alternative=greater)
  Mann-Whitney U:  U=177.0, p=0.03378 (alternative=greater)
  Rank-biserial effect size: +0.383

=== Check: do the two non-spatial groups differ from each other? ===

Nonspatial-verb vs Nonspatial-modifier:
  Welch's t-test:  t=-1.184, p=0.24641 (alternative=two-sided)
  Mann-Whitney U:  U=83.5, p=0.09722 (alternative=two-side

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
lo = min(spatial_sims.min(), verb_sims.min(), modifier_sims.min()) - 0.02
hi = max(spatial_sims.max(), verb_sims.max(), modifier_sims.max()) + 0.02
bins = np.linspace(lo, hi, 22)
ax.hist(verb_sims, bins=bins, alpha=0.5, label=f"non-spatial verb (n={len(verb_sims)})", color="tab:blue")
ax.hist(modifier_sims, bins=bins, alpha=0.5, label=f"non-spatial modifier (n={len(modifier_sims)})", color="tab:green")
ax.hist(spatial_sims, bins=bins, alpha=0.5, label=f"spatial (n={len(spatial_sims)})", color="tab:red")
for arr, color in [(verb_sims, "tab:blue"), (modifier_sims, "tab:green"), (spatial_sims, "tab:red")]:
    ax.axvline(arr.mean(), color=color, linestyle="--", linewidth=1)
ax.set_xlabel("cosine similarity within minimal pair")
ax.set_ylabel("count")
ax.set_title("CLIP text-encoder similarity: spatial vs. two non-spatial control groups")
ax.legend()
plt.tight_layout()
plt.savefig("01_clip_spatial_blindness_histogram.png", dpi=110)
plt.show()
print("saved 01_clip_spatial_blindness_histogram.png")

saved 01_clip_spatial_blindness_histogram.png


/var/folders/sb/x2_py571213939_gjl4zbhxc0000gn/T/ipykernel_26172/3274056985.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. How much of the corpus does this actually touch?

A blind spot in a rarely-used word class is a narrow footnote; a blind spot in a majority of
the corpus's captions is a limitation of the whole benchmark. Counted directly from the
materialized HumanML3D caption files this project already uses elsewhere
(`third_party/motion-diffusion-model/dataset/HumanML3D/texts/`), not estimated. Term list
expanded per review SUP-20260907-82 (which independently re-derived this count and found my
original list — `left/right/forward/backward/backwards/clockwise/counterclockwise/behind/in
front of` — was missing a few morphological variants: `forwards`, `anticlockwise`, `upleft`).

In [7]:
import glob
import re

TEXTS_DIR = os.path.join(MDM_ROOT, "dataset", "HumanML3D", "texts")
SPATIAL_TERMS = ["left", "right", "forward", "forwards", "backward", "backwards",
                  "clockwise", "counterclockwise", "anticlockwise", "upleft",
                  "behind", "in front of"]

txt_files = sorted(glob.glob(os.path.join(TEXTS_DIR, "*.txt")))
print(f"{len(txt_files)} caption files found under {TEXTS_DIR}")

total_captions = 0
spatial_captions = 0
term_counts = {t: 0 for t in SPATIAL_TERMS}

for path in txt_files:
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or "#" not in line:
                continue
            caption = line.split("#")[0].strip().lower()
            if not caption:
                continue
            total_captions += 1
            hit = False
            for term in SPATIAL_TERMS:
                if re.search(r"\b" + re.escape(term) + r"\b", caption):
                    term_counts[term] += 1
                    hit = True
            if hit:
                spatial_captions += 1

pct = 100 * spatial_captions / total_captions if total_captions else float("nan")
print(f"\nTotal captions scanned: {total_captions}")
print(f"Captions containing at least one spatial term: {spatial_captions} ({pct:.1f}%)")
print("\nPer-term counts (captions can contain more than one term):")
for term, count in sorted(term_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {term!r:20s} {count:6d}  ({100*count/total_captions:.2f}%)")

8198 caption files found under ../third_party/motion-diffusion-model/dataset/HumanML3D/texts



Total captions scanned: 24503
Captions containing at least one spatial term: 14112 (57.6%)

Per-term counts (captions can contain more than one term):
  'right'                5942  (24.25%)
  'forward'              5544  (22.63%)
  'left'                 5327  (21.74%)
  'backwards'             875  (3.57%)
  'in front of'           704  (2.87%)
  'clockwise'             633  (2.58%)
  'forwards'              346  (1.41%)
  'counterclockwise'      247  (1.01%)
  'backward'              243  (0.99%)
  'behind'                148  (0.60%)
  'anticlockwise'          17  (0.07%)
  'upleft'                  9  (0.04%)


## 5. Verdict

Stated directly from the numbers computed above, not asserted in advance. The load-bearing test
is spatial-vs-modifier (the same syntactic slot); the verb comparison is reported for continuity
but does not by itself distinguish "spatial blindness" from "modifiers separate worse than
verbs.

In [8]:
verdict_lines = []

if spatial_sims.mean() > modifier_sims.mean() and u_p_modifier < 0.05:
    verdict_lines.append(
        f"CONFIRMED, isolated from the verb-vs-modifier confound: spatial minimal pairs are "
        f"measurably LESS separated (mean cos-sim {spatial_sims.mean():.4f}) than non-spatial "
        f"MODIFIER pairs in the same syntactic slot (mean cos-sim {modifier_sims.mean():.4f}), "
        f"Mann-Whitney p={u_p_modifier:.5f}, rank-biserial effect size {rb_modifier:+.3f}. "
        f"This is not merely 'modifiers separate worse than verbs' -- the modifier-vs-modifier "
        f"comparison isolates spatial-ness specifically as the source of the effect."
    )
elif u_p_verb_vs_mod < 0.05:
    verdict_lines.append(
        f"NOT ISOLATED: spatial mean {spatial_sims.mean():.4f} vs modifier mean "
        f"{modifier_sims.mean():.4f} (p={u_p_modifier:.5f}) does not clear significance, but the "
        f"two non-spatial groups themselves differ (verb mean {verb_sims.mean():.4f} vs modifier "
        f"mean {modifier_sims.mean():.4f}, p={u_p_verb_vs_mod:.5f}). The honest headline here is "
        f"'CLIP under-separates modifiers generally, spatial included' -- a real limitation for "
        f"motion conditioning, just a different and less specific one than 'spatial blindness.'"
    )
else:
    verdict_lines.append(
        f"NOT CONFIRMED at this n: spatial mean {spatial_sims.mean():.4f} vs modifier mean "
        f"{modifier_sims.mean():.4f}, Mann-Whitney p={u_p_modifier:.5f}. Do not report a spatial "
        f"blind spot from this notebook's data if this branch printed."
    )

verdict_lines.append(
    f"\n(For reference, the original verb-comparison result: spatial mean {spatial_sims.mean():.4f} "
    f"vs non-spatial-verb mean {verb_sims.mean():.4f}, p={u_p_verb:.5f}, effect size {rb_verb:+.3f} "
    f"-- consistent with, but not sufficient on its own to isolate, the spatial-specific claim above.)"
)

verdict_lines.append(
    f"\nCorpus footprint: {pct:.1f}% of {total_captions} HumanML3D captions contain at least one "
    f"of {SPATIAL_TERMS}. " +
    ("This is a narrow slice of the benchmark, not a dominant failure mode." if pct < 10 else
     "This is not a narrow footnote -- a majority of the benchmark's own captions use language "
     "this encoder may not distinguish.")
)

verdict_lines.append(
    "\nCaveat this project's own discipline requires stating: n=%d pairs per group is small. "
    "The Mann-Whitney test does not assume normality, which matters at this n, but it does not "
    "make 16 pairs a large sample -- this establishes a real, measured effect at the pairs tested, "
    "not a guarantee it holds for every spatial construction in the language." % len(spatial_sims)
)

print("\n".join(verdict_lines))

CONFIRMED, isolated from the verb-vs-modifier confound: spatial minimal pairs are measurably LESS separated (mean cos-sim 0.9654) than non-spatial MODIFIER pairs in the same syntactic slot (mean cos-sim 0.9446), Mann-Whitney p=0.03378, rank-biserial effect size +0.383. This is not merely 'modifiers separate worse than verbs' -- the modifier-vs-modifier comparison isolates spatial-ness specifically as the source of the effect.

(For reference, the original verb-comparison result: spatial mean 0.9654 vs non-spatial-verb mean 0.9296, p=0.00056, effect size +0.680 -- consistent with, but not sufficient on its own to isolate, the spatial-specific claim above.)

Corpus footprint: 57.6% of 24503 HumanML3D captions contain at least one of ['left', 'right', 'forward', 'forwards', 'backward', 'backwards', 'clockwise', 'counterclockwise', 'anticlockwise', 'upleft', 'behind', 'in front of']. This is not a narrow footnote -- a majority of the benchmark's own captions use language this encoder may n